# Restitution de graphe orienté avec networkx + pyvis

Ce notebook construit un graphe orienté avec **networkx** à partir d'un
fichier texte (format `NOEUDS`/`ARETES`, identique au reste du projet), puis
le restitue avec **pyvis** (bibliothèque Python pour [vis.js](https://visjs.org/)) :
zoomable, déplaçable, **clique un nœud** pour surligner ses
prédécesseurs/successeurs directs, **survole** pour le détail. Les arêtes qui
referment un cycle sont en rouge pointillé.

**Différence avec `python-notebook/`** (la variante plotly) : pyvis écrit un
vrai fichier `.html` autonome par graphe (ex. `graph_exemple.html`), affiché
ici via `<iframe src="...">`. Comme le script d'interactivité vit dans ce
fichier séparé et pas directement dans la sortie de la cellule, Jupyter n'a pas
besoin qu'on marque le notebook comme "de confiance" (*Trust Notebook*) pour
que le clic fonctionne — un souci qu'on peut rencontrer avec `python-notebook/`.

matplotlib/seaborn ont été écartés : dans un notebook, ils ne produisent
qu'une image statique, sans aucune interaction possible dessus.


## 1. Fonctions (parsing, détection de cycle, rendu pyvis)


In [1]:
# --- Imports -----------------------------------------------------------
from pathlib import Path      # manipulation de chemins de fichiers
import json                   # pour transformer des données Python en JSON (config vis.js)

import networkx as nx         # graphe orienté : parsing, prédécesseurs/successeurs
from pyvis.network import Network  # génère un vrai fichier HTML autonome (vis.js) par graphe
from IPython.display import IFrame, display  # afficher un fichier HTML / un widget dans une cellule
import ipywidgets as widgets  # bouton d'upload de fichier (section 4)

# En-têtes de section reconnus dans le fichier .txt (avec ou sans accent)
NODE_HEADERS = {"noeuds", "nœuds", "nodes"}
EDGE_HEADERS = {"aretes", "arêtes", "edges"}


def parse_graph(text: str) -> nx.DiGraph:
    """Parse le format NOEUDS/ARETES et construit un graphe orienté networkx."""
    graph = nx.DiGraph()
    section = None  # None, "nodes" ou "edges" selon la ligne d'en-tête vue en dernier

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        header = line.lower()
        if header in NODE_HEADERS:
            section = "nodes"
            continue
        if header in EDGE_HEADERS:
            section = "edges"
            continue

        if section == "nodes":
            graph.add_node(line)
        elif section == "edges":
            parts = line.split()
            if len(parts) < 2:
                raise ValueError(f"Ligne d'arête invalide (attendu 'source cible'): {raw_line!r}")
            graph.add_edge(parts[0], parts[1])
        else:
            raise ValueError(f"Ligne hors section (attendu NOEUDS/ARETES avant tout contenu): {raw_line!r}")

    if graph.number_of_nodes() == 0:
        raise ValueError("Aucun noeud trouvé dans le fichier.")

    return graph


def find_back_edges(graph: nx.DiGraph) -> set[tuple[str, str]]:
    """DFS classique : une arête vers un nœud déjà sur la pile d'appel referme un cycle."""
    back_edges: set[tuple[str, str]] = set()
    visited: set[str] = set()
    on_stack: set[str] = set()

    def dfs(node: str) -> None:
        visited.add(node)
        on_stack.add(node)
        for succ in graph.successors(node):
            if succ in on_stack:
                back_edges.add((node, succ))
            elif succ not in visited:
                dfs(succ)
        on_stack.discard(node)

    for node in graph.nodes:
        if node not in visited:
            dfs(node)

    return back_edges


# Couleurs des nœuds selon leur état (mêmes couleurs que python-mvc/ et python-notebook/,
# pour rester cohérent visuellement entre les différentes variantes du projet)
_DEFAULT_COLOR = "#4C72B0"
_SELECTED_COLOR = "#f59e0b"
_PREDECESSOR_COLOR = "#0ea5e9"
_SUCCESSOR_COLOR = "#16a34a"
_DIMMED_COLOR = "#cbd5e1"
_CYCLE_EDGE_COLOR = "#dc2626"
_EDGE_COLOR = "#94a3b8"

# Le script injecté ci-dessous tourne dans le navigateur (JavaScript, pas Python) : il
# réagit au clic sur un nœud pour surligner ses prédécesseurs/successeurs directs.
# vis-network (la bibliothèque utilisée par pyvis) expose déjà les variables globales
# "network", "nodes" et "edges" dans le HTML généré : on les réutilise directement,
# pas besoin de les recréer. Contrairement à la version plotly (python-notebook/), les
# arêtes de cycle peuvent être en vrais pointillés ici (vis.js supporte "dashes").
_CLICK_SCRIPT_TEMPLATE = """
<script type="text/javascript">
(function() {
  var DEFAULT = "%(default)s", SELECTED = "%(selected)s", PRED = "%(pred)s",
      SUCC = "%(succ)s", DIM = "%(dim)s", EDGE = "%(edge)s", CYCLE = "%(cycle)s";
  var selectedId = null;

  // style "de base" d'une arête : rouge pointillé si elle referme un cycle, gris sinon
  function baseEdgeStyle(edge) {
    if (edge.cycle) return {color: CYCLE, dashes: true, width: 3};
    return {color: EDGE, dashes: false, width: 1};
  }

  function clearSelection() {
    nodes.update(nodes.getIds().map(function(id) { return {id: id, color: DEFAULT}; }));
    edges.update(edges.getIds().map(function(id) {
      var style = baseEdgeStyle(edges.get(id));
      return {id: id, color: style.color, dashes: style.dashes, width: style.width};
    }));
    selectedId = null;
  }

  function selectNode(nodeId) {
    // getConnectedEdges : tous les ids d'arêtes touchant ce nœud (entrantes + sortantes)
    var connected = network.getConnectedEdges(nodeId);
    var predIds = [], succIds = [];
    connected.forEach(function(edgeId) {
      var e = edges.get(edgeId);
      if (e.to === nodeId && e.from !== nodeId) predIds.push(e.from);
      if (e.from === nodeId && e.to !== nodeId) succIds.push(e.to);
    });

    nodes.update(nodes.getIds().map(function(id) {
      if (id === nodeId) return {id: id, color: SELECTED};
      if (predIds.indexOf(id) !== -1) return {id: id, color: PRED};
      if (succIds.indexOf(id) !== -1) return {id: id, color: SUCC};
      return {id: id, color: DIM};
    }));
    edges.update(edges.getIds().map(function(id) {
      if (connected.indexOf(id) !== -1) return {id: id, color: SELECTED, width: 3};
      var style = baseEdgeStyle(edges.get(id));
      return {id: id, color: DIM, dashes: style.dashes, width: style.width};
    }));
    selectedId = nodeId;
  }

  network.on("click", function(params) {
    if (params.nodes.length > 0) {
      var clickedId = params.nodes[0];
      if (selectedId === clickedId) { clearSelection(); } else { selectNode(clickedId); }
    } else {
      clearSelection();
    }
  });
})();
</script>
"""


def render_directed_graph(graph, output_path, height="600px"):
    """Écrit un fichier HTML autonome (vis.js) pour ce graphe, et renvoie un IFrame
    pour l'afficher directement sous la cellule.

    Zoomable/déplaçable nativement (vis.js). Clique un nœud pour surligner ses
    prédécesseurs (bleu) et successeurs (vert) directs, reclique pour désélectionner.
    Survole un nœud pour son détail (infobulle). Les arêtes de cycle sont en rouge
    pointillé.

    cdn_resources="in_line" embarque vis.js directement dans le fichier généré (comme
    plotly.js dans python-notebook/) : le fichier reste utilisable hors ligne, et comme
    c'est un vrai fichier séparé affiché via <iframe src="...">, Jupyter n'a pas besoin
    de faire confiance ("Trust Notebook") à ce script pour l'exécuter — contrairement à
    du HTML/JS injecté directement dans la sortie d'une cellule (cas de python-notebook/).
    """
    back_edges = find_back_edges(graph)

    net = Network(height=height, width="100%", directed=True, notebook=True, cdn_resources="in_line")

    for n in graph.nodes:
        preds = sorted(graph.predecessors(n)) or ["—"]
        succs = sorted(graph.successors(n)) or ["—"]
        title = f"<b>{n}</b><br>prédécesseurs: {', '.join(preds)}<br>successeurs: {', '.join(succs)}"
        net.add_node(n, label=n, title=title, color=_DEFAULT_COLOR)

    for u, v in graph.edges:
        is_cycle_edge = (u, v) in back_edges
        net.add_edge(
            u, v, arrows="to",
            color=_CYCLE_EDGE_COLOR if is_cycle_edge else _EDGE_COLOR,
            dashes=is_cycle_edge, width=3 if is_cycle_edge else 1,
            cycle=is_cycle_edge,  # propriété perso (pas standard vis.js), relue par le script de clic
        )

    # layout hiérarchique intégré à vis.js (façon Sugiyama) : pas besoin de calculer
    # les positions nous-mêmes comme dans python-notebook/ (layered_positions)
    net.set_options(json.dumps({
        "layout": {"hierarchical": {"enabled": True, "direction": "UD", "sortMethod": "directed",
                                     "nodeSpacing": 130, "levelSeparation": 110}},
        "physics": {"enabled": False},
        "interaction": {"hover": True},
        "edges": {"smooth": {"type": "cubicBezier", "forceDirection": "vertical"}},
    }))

    html = net.generate_html(notebook=True)
    click_script = _CLICK_SCRIPT_TEMPLATE % {
        "default": _DEFAULT_COLOR, "selected": _SELECTED_COLOR, "pred": _PREDECESSOR_COLOR,
        "succ": _SUCCESSOR_COLOR, "dim": _DIMMED_COLOR, "edge": _EDGE_COLOR, "cycle": _CYCLE_EDGE_COLOR,
    }
    html = html.replace("</body>", click_script + "\n</body>")
    Path(output_path).write_text(html, encoding="utf-8")

    return IFrame(output_path, width="100%", height=height)


## 2. Petit graphe (`exemple.txt`)

5 nœuds, 6 arêtes, avec un cycle (`E -> A`, en rouge pointillé ci-dessous).


In [2]:
text = Path("../exemple.txt").read_text(encoding="utf-8")
graph = parse_graph(text)
print(f"{graph.number_of_nodes()} nœuds, {graph.number_of_edges()} arêtes, "
      f"cycle détecté : {len(find_back_edges(graph)) > 0}")

render_directed_graph(graph, "graph_exemple.html")


5 nœuds, 6 arêtes, cycle détecté : True


## 3. Grand graphe (`exemple_50_noeuds.txt`)

Même fonction, juste un autre fichier — pratique pour voir comment pyvis
(zoom, pan, glisser les nœuds) se comporte sur un graphe plus fourni.


In [3]:
text_50 = Path("../exemple_50_noeuds.txt").read_text(encoding="utf-8")
graph_50 = parse_graph(text_50)
print(f"{graph_50.number_of_nodes()} nœuds, {graph_50.number_of_edges()} arêtes")

render_directed_graph(graph_50, "graph_50.html", height="700px")


50 nœuds, 100 arêtes


## 4. Upload interactif d'un fichier `.txt`

Plutôt qu'un chemin de fichier codé en dur, un bouton d'upload : choisis un
fichier au format `NOEUDS`/`ARETES` depuis ton poste, le graphe correspondant
s'affiche automatiquement en dessous (écrit dans `graph_upload.html`).

⚠️ Le bouton lui-même passe par `ipywidgets` et **nécessite un noyau Python
actif** (Jupyter, DataLab...) — dans l'export statique `index.html`, le
bouton s'affiche mais l'upload ne déclenche rien (aucun noyau pour le
traiter). Une fois le graphe généré en revanche, son interactivité (clic,
survol) fonctionne partout, y compris dans `index.html` — c'est l'avantage de
pyvis évoqué au tout début du notebook.


In [4]:
# Crée le bouton d'upload : accepte uniquement les fichiers .txt, un seul à la fois
upload_widget = widgets.FileUpload(accept=".txt", multiple=False, description="Choisir un fichier .txt")
# Output() = une zone de la page où afficher du texte/graphique par programme
upload_output = widgets.Output()


def _handle_upload(change):
    """Appelée automatiquement dès qu'un fichier est choisi dans le bouton (voir
    upload_widget.observe(...) tout en bas)."""
    with upload_output:
        upload_output.clear_output()
        if not upload_widget.value:
            print("Aucun fichier.")
            return
        uploaded = upload_widget.value[0]  # ipywidgets>=8 : dict avec name/type/size/content/last_modified

        # try/except : message clair en français plutôt qu'un traceback si le fichier
        # n'est pas au bon format (mauvais encodage, sections NOEUDS/ARETES manquantes...)
        try:
            text = bytes(uploaded["content"]).decode("utf-8")
            graph = parse_graph(text)
            cycle = len(find_back_edges(graph)) > 0
        except UnicodeDecodeError:
            print(f"Fichier invalide ({uploaded['name']}) : ce n'est pas un fichier texte UTF-8.")
            return
        except ValueError as exc:
            print(f"Fichier invalide ({uploaded['name']}) : {exc}")
            return

        print(f"{uploaded['name']} : {graph.number_of_nodes()} nœuds, {graph.number_of_edges()} arêtes, cycle détecté : {cycle}")
        display(render_directed_graph(graph, "graph_upload.html"))


upload_widget.observe(_handle_upload, names="value")
display(upload_widget, upload_output)

FileUpload(value=(), accept='.txt', description='Choisir un fichier .txt')

Output()